# Summarizer - User Interface - Proof of Concept

In [1]:
%load_ext autoreload

%autoreload 2

In [2]:
import json
from datetime import date, datetime
from functools import partial
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
from tqdm.auto import tqdm

from llm import NoRelevantDataFoundError, extract_from_llm_output, openai_llm, rag
from vectordb import MilvusClientFix, milvus_search


In [3]:
nb_path = Path()

In [4]:
load_dotenv(nb_path / "../.env", verbose=True)

False

## Configuration and initialization

In [5]:
hn_dump_file = "hn_news.json"
lr_dump_file = "lr_news.json"

collection_name = "llm_summarizer_poc"
collection_db_path = "./milvus_summarizer.db"

In [6]:
milvus_client = MilvusClientFix.get_instance(collection_db_path)

In [7]:
openai_client = OpenAI()

In [8]:
milvus_search_fn = partial(milvus_search, milvus_client, collection_name)

In [9]:
openai_llm_fn = partial(openai_llm, openai_client)

In [10]:
def build_summary_prompt(query: str, search_results: list[dict]) -> str:
    prompt_template = """
You're the skilled specialist. Summarize the most important points from the CONTEXT that might be useful or interesting for a specialist and related to  QUERY. 
Use only the facts from the CONTEXT when finding relevancy but provide some comparative summary with the state-of-the-arts if possible.
If the context fragment does not have close relation to the query, provide a short note why a fragment is not relevant.
Provide the output as JSON with the list of dictionaries with the following fields: fragment_id, summary, is_relevant. Value in is_relevant should be True if the fragment is relevant to the KEYWORDS and False otherwise.

QUERY: {query}

CONTEXT: 
{context}
""".strip()

    context = ""
    
    for idx, doc in enumerate(search_results):
        context = context + f"FRAGMENT_{doc['document_uid']}: {doc['text']}\n\n"
    
    prompt = prompt_template.format(query=query, context=context).strip()
    return prompt


In [11]:
query = "Data Engineering"
start_date = None
end_date = None

In [12]:
rag_summary = rag(query, build_summary_prompt, openai_llm_fn, milvus_search_fn, start_dt=start_date, end_dt=end_date)

In [13]:
rag_summary

'```json\n[\n    {\n        "fragment_id": "FRAGMENT_4af46c0e54",\n        "summary": "The blog entry discusses the author\'s work on sending an Ethernet packet using an STM32F401 microcontroller. The focus is on building a TCP/IP stack from scratch and discusses Technical aspects of Ethernet including its technologies, standards, and hardware requirements like ASICs and SPI communication with the W5100 Ethernet chip. The project represents a hands-on approach to low-level networking, which is crucial in data engineering contexts where understanding networking fundamentals impacts data flow and processing efficiency.",\n        "is_relevant": true\n    },\n    {\n        "fragment_id": "FRAGMENT_a4ff470e6b",\n        "summary": "This fragment discusses various cybersecurity topics and vulnerabilities, including exploits, software vulnerabilities, and security frameworks, which, while relevant to software engineering, do not directly pertain to the core concepts of data engineering spec

For every relevant fragment find most related documents from the whole history and provide a perspective on the topic.


In [14]:
rag_cleaned = extract_from_llm_output(rag_summary)
rag_cleaned

[{'fragment_id': 'FRAGMENT_4af46c0e54',
  'summary': "The blog entry discusses the author's work on sending an Ethernet packet using an STM32F401 microcontroller. The focus is on building a TCP/IP stack from scratch and discusses Technical aspects of Ethernet including its technologies, standards, and hardware requirements like ASICs and SPI communication with the W5100 Ethernet chip. The project represents a hands-on approach to low-level networking, which is crucial in data engineering contexts where understanding networking fundamentals impacts data flow and processing efficiency.",
  'is_relevant': True},
 {'fragment_id': 'FRAGMENT_a4ff470e6b',
  'summary': 'This fragment discusses various cybersecurity topics and vulnerabilities, including exploits, software vulnerabilities, and security frameworks, which, while relevant to software engineering, do not directly pertain to the core concepts of data engineering specifically around data processing or architecture.',
  'is_relevant': 

In [15]:
from more_itertools import chunked

MAX_SCOPE = 100
BATCH_SIZE = 10

def rag_batched(
    query: str,
    prompt_fn: callable,
    llm_fn: callable,
    search_fn: callable,
    num_results: int = MAX_SCOPE,
    batch_size: int = BATCH_SIZE,
    start_dt: datetime | None = None,
    end_dt: datetime | None = None
) -> list[dict]:
    """Return relevant answers built using RAG.
    
    The goal is to find relevant documents and disregard irrelevant ones.
    Take a lot of documents, split them into batches. Assume that documents are ordered by "distance" between embeddings. If two consecutive batches do not contain relevant fragments, stop the process.
    
    LLM Response should contain a list of dictionaries with at least "is_relevant" field. 
    """
    search_results = search_fn(
        query=query,
        num_results=num_results,
        start_dt=start_dt,
        end_dt=end_dt
    )
    
    if not search_results:
        raise NoRelevantDataFoundError("No relevant results found.")
    
    prev_batch_relevant = True
    relevant_results = []
    for batch in tqdm(chunked(search_results, batch_size)):
        prompt = prompt_fn(query, batch)
        answer = llm_fn(prompt)
        
        cleaned = extract_from_llm_output(answer)
        count_relevant = 0
        for item in cleaned:
            if item["is_relevant"]:
                relevant_results.append(item)
                count_relevant += 1
                
        if count_relevant == 0:
            if not prev_batch_relevant:
                break
            
            prev_batch_relevant = False

    return relevant_results

In [16]:
extended_summary = rag_batched(query, build_summary_prompt, openai_llm_fn, milvus_search_fn, start_dt=start_date, end_dt=end_date)

extended_summary

0it [00:00, ?it/s]

JSONDecodeError: Expecting value: line 5 column 24 (char 500)

In [ ]:
len(extended_summary)

39

In [17]:
# add original references to the data found
def format_extended_summary(query: str, extended_summary: list[dict], original_data: list[dict]) -> str:
    """Pretty print the extended summary."""
    if not extended_summary:
        out = f"Query: **{query}**\n\nNo relevant data found."
        return out
    
    out = f"Query: **{query}**\n\nThe following posts found:\n\n"
    
    urls = []
    
    for entry in extended_summary:
        doc_uid = entry["fragment_id"]
        if doc_uid.startswith("FRAGMENT_"):
            doc_uid = doc_uid.removeprefix("FRAGMENT_")
        summary = entry["summary"]
        
        # add from original data
        for doc in original_data:
            if doc["document_uid"] == doc_uid:
                ref = doc["url"]
                title = doc["title"]
                if ref not in urls:
                    urls.append(ref)
                    out += f"[{title}]({ref})\n\n{summary}\n\n"
                
                break

    return out


In [18]:
def load_stored(file_path: str) -> list:
    stored = []
    try:
        with open(file_path, "r") as fp:
            stored = json.load(fp)
    except (FileNotFoundError, json.JSONDecodeError):
        pass
    
    return stored

stored_data = load_stored(hn_dump_file) + load_stored(lr_dump_file)

## The working example

In [27]:
query = "small language model"
# start_dt = datetime(2024, 10, 28, 0, 0)
# end_dt = datetime(2024, 11, 1, 0, 0)
start_dt = None
end_dt = None

In [28]:
from IPython.display import display, Markdown

out = format_extended_summary(
    query,
    rag_batched(query, build_summary_prompt, openai_llm_fn, milvus_search_fn, start_dt=start_dt, end_dt=end_dt),
    stored_data
)

display(Markdown(out))

0it [00:00, ?it/s]

JSONDecodeError: Expecting value: line 5 column 24 (char 407)